## ცხრილების ჯაჭვი

```
getdata.raw
│
├── raw_peaks ─────────┐
│                      ▼
├── raw_expeditions ─> [1] CalcPeaks ──┬──> [2] CalcExpeditions
│                                      │
├── raw_members ───────────────────────┴──> [3] CalcMembers
│                                                   │
│                                                   ├──> [5] CalcPeakRisk
│                                                   │         │
│                                                   │         ▼
│                                                   │    [6] CalcPeakResidual 
│                                                   │
│                                                   ├──> [7] CalcNationProfile
│                                                   │
│                                                   └──> [8] CalcExpeditionTrend
│
└── raw_members + CalcPeaks ──────────────────> [9] CalcAltitudeGroupRisk

                        [4] CalcOxygenCurve  (ფორმულით გენერირებული,
                                              მონაცემებზე არ არის დამოკიდებული)
```


## 1. CalcPeaks — მწვერვალების განზომილება

**შემოდის:** `raw_peaks` 468 სტრიქონი

**გამოდის:** `CalcPeaks`  468 სტრიქონი *(ფილტრი არ არის)*

### რა კეთდება

- `'NA'` → `NULL`: `peak_alternative_name` (223), `first_ascent_year` (132), `first_ascent_country` (132)
- `FirstAscentYear` გადაჰყავს `INT`-ში
- ამოღებულია `first_ascent_expedition_id` — არცერთ ვიზუალში არ გვჭირდება

### ჟანგბადის თეორიული გამოთვლა

აქ ითვლება ანალიზის **მთავარი ცვლადი** — ბაროსტატიკური ფორმულით:

$$P(h) / P_0 = e^{-h/H}$$

სადაც $H = RT/(Mg) \approx 7300$ მ — ჟანგბადის მასშტაბური სიმაღლე.

| მწვერვალი | სიმაღლე | `OxygenAvailability` |
|---|---|---|
| ევერესტი | 8,850 მ | ~0.298 (30%) |
| ჩო-ოიუ | 8,188 მ | ~0.327 (33%) |
| ანაპურნა I | 8,091 მ | ~0.331 (33%) |

**`AltitudeGroup`** ყველა 468 მწვერვალს ჯგუფავს — ის მხოლოდ `CalcAltitudeGroupRisk`-ში გამოიყენება, სკოუპის დასასაბუთებლად.

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcPeaks AS
SELECT
    Pk.peak_id AS PeakId,
    Pk.peak_name AS PeakName,
    NULLIF(Pk.peak_alternative_name, 'NA') AS PeakAlternativeName,
    CAST(Pk.height_metres AS INT) AS HeightMetres,
    Pk.climbing_status AS ClimbingStatus, 
    CAST(NULLIF(Pk.first_ascent_year, 'NA') AS INT) AS FirstAscentYear,
    NULLIF(Pk.first_ascent_country, 'NA') AS FirstAscentCountry,
    ROUND(EXP(-CAST(Pk.height_metres AS DOUBLE) / 7300.0), 4) AS OxygenAvailability,
    -- ჟანგბადის თეორიული წილი ზღვის დონესთან შედარებით 0-1
    -- სიმაღლის ფართო კატეგორია (ყველა 468 მწვერვალი).
    -- გამოიყენება სკოუპის დასასაბუთებლად calc_altitude_group_riskში
    CASE
        WHEN pk.height_metres >= 8000 THEN '8000+ (სიკვდილის ზონა)'
        WHEN pk.height_metres >= 7500 THEN '7500-7999'
        WHEN pk.height_metres >= 7000 THEN '7000-7499'
        WHEN pk.height_metres >= 6500 THEN '6500-6999'
        ELSE '6500-ზე დაბალი'
    END AS AltitudeGroup
FROM getdata.raw.raw_peaks AS Pk;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcExpeditions AS
SELECT 
    Ex.expedition_id AS ExpeditionId,
    Ex.peak_id AS PeakId,
    Pk.PeakName,
    Pk.HeightMetres,
    Pk.OxygenAvailability,
    CAST(Ex.Year AS INT) AS ExpeditionYear,
        -- ათწლეული ტრენდის ანალიზისთვის
    CAST(FLOOR(CAST(Ex.Year AS INT) / 10) * 10 AS INT) AS Decade,
    NULLIF(Ex.Season, 'Unknown') AS Season,
    CAST(NULLIF(Ex.highpoint_metres, 'NA') AS INT) AS HighpointMetres,
    Ex.termination_reason AS TerminationReason,
    -- წარმატება = მთავარ მწვერვალზე ასვლა
    CASE
       WHEN Ex.termination_reason LIKE 'Success%' THEN TRUE
       ELSE FALSE
    END AS IsSuccess,
    CAST(Ex.members AS INT) AS MemberCount,
    CAST(Ex.member_deaths AS INT) AS MemberDeathCount,
    CAST(Ex.hired_staff AS INT) AS HiredCount,
    CAST(Ex.hired_staff_deaths AS INT) AS HiredDeathCount,
    CAST(Ex.oxygen_used AS BOOLEAN) AS UsedOxygen,
    NULLIF(Ex.trekking_agency, 'NA') AS TrekkingAgency
FROM getdata.raw.raw_expeditions AS Ex
INNER JOIN getdata.calculated.CalcPeaks AS Pk
 ON Ex.peak_id = Pk.PeakId
WHERE Pk.HeightMetres >= 8000;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcMembers AS
SELECT 
    Mb.member_id AS MemberId,
    Mb.expedition_id AS ExpeditionId,
    Mb.peak_id AS PeakId,
    Pk.PeakName,
    Pk.HeightMetres,
    Pk.OxygenAvailability,
    CAST(Mb.Year AS INT) AS ExpeditionYear,
    CAST(FLOOR(CAST(Mb.Year AS INT) / 10) * 10 AS INT) AS Decade,
    NULLIF(Mb.season, 'Unknown') AS Season,
    NULLIF(Mb.Sex, 'NA') AS Sex,
    CAST(NULLIF(Mb.age, 'NA') AS INT) AS Age,
    NULLIF(Mb.citizenship, 'NA') AS Citizenship,
    NULLIF(Mb.expedition_role, 'NA') AS ExpeditionRole,
    CAST(Mb.hired AS BOOLEAN) AS IsHiredStaff,
    CAST(Mb.Success AS BOOLEAN) AS ReachedSummit,
    CAST(Mb.died AS BOOLEAN) AS Died,
    CAST(Mb.Solo AS BOOLEAN) AS WasSolo,
    CAST(Mb.oxygen_used AS BOOLEAN) AS UsedOxygen,
    CAST(NULLIF(Mb.highpoint_metres, 'NA') AS INT) AS HighpointMetres,
    NULLIF(Mb.death_cause, 'NA') AS DeathCause
FROM getdata.raw.raw_members AS Mb
INNER JOIN getdata.calculated.CalcPeaks AS Pk
 ON Mb.peak_id = Pk.PeakId
WHERE Pk.HeightMetres >= 8000;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.calcOxygenCurve AS
SELECT
    curve.HeightMetres,
    ROUND(EXP(-CAST(curve.HeightMetres AS DOUBLE) / 7300.0), 4) AS OxygenAvailability,
    ROUND(EXP(-CAST(curve.HeightMetres AS DOUBLE) / 7300.0) * 100, 2) AS OxygenPCTOfSeaLevel,
    -- 8000 მ ზემოთ სხეული ვეღარ ადაპტირდება
    CASE
        WHEN curve.HeightMetres >= 8000 THEN TRUE
        ELSE FALSE
    END  AS IsDeathZone
FROM (
    SELECT EXPLODE(SEQUENCE(0, 9000, 100)) AS HeightMetres
) AS curve;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcPeakRisk AS
SELECT
    Mb.PeakId,
    Mb.PeakName,
    Mb.HeightMetres,
    Mb.OxygenAvailability,
    COUNT(*) AS ClimberCount,
    SUM(CASE WHEN Mb.Died THEN 1 ELSE 0 END) AS DeathCount,
    SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) AS SummitCount,
    ROUND(100.0 * SUM(CASE WHEN Mb.died THEN 1 ELSE 0 END) / COUNT(*), 3)
                                                                      AS DeathRatePCT,
    ROUND(100.0 * SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                                      AS SummitRatePCT
FROM getdata.calculated.CalcMembers AS Mb
GROUP BY
    Mb.PeakId,
    Mb.PeakName,
    Mb.HeightMetres,
    Mb.OxygenAvailability;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcPeakResidual AS
WITH ModelInput AS (
    SELECT
        Pr.OxygenAvailability,
        Pr.DeathRatePCT
    FROM getdata.calculated.CalcPeakRisk AS Pr
    WHERE Pr.ClimberCount >= 100
),
ModelCoefficients AS (
    SELECT
        COVAR_POP(Mi.DeathRatePCT, Mi.OxygenAvailability)
            / VAR_POP(Mi.OxygenAvailability) AS Slope,
        AVG(Mi.DeathRatePCT)
            - (COVAR_POP(Mi.DeathRatePCT, Mi.OxygenAvailability)
            / VAR_POP(Mi.OxygenAvailability))
            * AVG(Mi.OxygenAvailability) AS Intercept,
        POW(CORR(Mi.DeathRatePCT, Mi.OxygenAvailability), 2) AS RSquared
    FROM ModelInput AS Mi
)
SELECT
    Pr.PeakId,
    Pr.PeakName,
    Pr.HeightMetres,
    Pr.OxygenAvailability,
    Pr.ClimberCount,
    Pr.DeathCount,
    Pr.DeathRatePCT AS ActualDeathRatePCT,
    ROUND(Mc.Intercept
        + Mc.Slope * Pr.OxygenAvailability, 3) AS ExpectedDeathRatePCT,
    ROUND(Pr.DeathRatePCT
        - (Mc.Intercept + Mc.Slope * Pr.OxygenAvailability), 3) AS ResidualDeathRatePCT,
    ROUND(Mc.RSquared, 4) AS ModelRSquared,
    CASE
        WHEN Pr.ClimberCount >= 100 THEN TRUE
        ELSE FALSE
    END AS InModel,
    CASE
        WHEN Pr.ClimberCount < 100
            THEN 'მცირე შერჩევა (n<100)'
        WHEN Pr.DeathRatePCT
            > Mc.Intercept + Mc.Slope * Pr.OxygenAvailability
            THEN 'მოსალოდნელზე სახიფათო'
        ELSE 'მოსალოდნელზე უსაფრთხო'
    END AS RiskVerdict,
    CASE
        WHEN Pr.HeightMetres >= 8500 THEN '8500+'
        WHEN Pr.HeightMetres >= 8200 THEN '8200-8499'
        ELSE '8000-8199'
    END AS HeightBand
FROM getdata.calculated.CalcPeakRisk AS Pr
CROSS JOIN ModelCoefficients AS Mc;


In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcNationProfile AS
SELECT
    mb.Citizenship AS Nation,
    COUNT(*) AS ClimberCount,
    SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) AS SummitCount,
    SUM(CASE WHEN Mb.Died THEN 1 ELSE 0 END) AS DeathCount,
    ROUND(100.0 * SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) / COUNT(*), 2) AS SummitRatePCT,
    ROUND(100.0 * SUM(CASE WHEN Mb.died THEN 1 ELSE 0 END) / COUNT(*), 3) AS DeathRatePCT,
    SUM(CASE WHEN Mb.IsHiredStaff THEN 1 ELSE 0 END) AS HiredStaffCount
FROM getdata.calculated.CalcMembers AS Mb
WHERE mb.citizenship IS NOT NULL
GROUP BY mb.citizenship;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcExpeditionTrend AS
SELECT
    Mb.ExpeditionYear,
    Mb.UsedOxygen,
    COUNT(DISTINCT Mb.ExpeditionId) AS ExpeditionCount,
    COUNT(*) AS ClimberCount,
    SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) AS SummitCount,
    SUM(CASE WHEN Mb.Died THEN 1 ELSE 0 END) AS DeathCount,
    ROUND(100.0 * SUM(CASE WHEN Mb.ReachedSummit THEN 1 ELSE 0 END) / COUNT(*), 2) AS SummitRatePCT,
    ROUND(100.0 * SUM(CASE WHEN Mb.Died THEN 1 ELSE 0 END) / COUNT(*), 3) AS DeathRatePCT
FROM getdata.calculated.CalcMembers AS Mb
GROUP BY
    Mb.ExpeditionYear,
    Mb.UsedOxygen;

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.CalcAltitudeGroupRisk AS
SELECT
    Pk.AltitudeGroup,
    MIN(Pk.HeightMetres) AS MinHeightMetres,
    MAX(Pk.HeightMetres) AS MaxHeightMetres,
    COUNT(DISTINCT Mb.peak_id) AS PeakCount,
    COUNT(*) AS ClimberCount,
    SUM(CASE WHEN CAST(Mb.Died AS BOOLEAN) THEN 1 ELSE 0 END) AS DeathCount,
    ROUND(100.0 * SUM(CASE WHEN CAST(Mb.Died AS BOOLEAN) THEN 1 ELSE 0 END) / COUNT(*), 2) AS DeathRatePCT
FROM getdata.raw.raw_members AS Mb
INNER JOIN getdata.calculated.CalcPeaks AS Pk
        ON Mb.peak_id = Pk.PeakId
GROUP BY Pk.AltitudeGroup;
